# Import libraries

In [1]:
import pypsa
import yaml
import pandas as pd
import geopandas as gpd
import numpy as np
import scipy as sp
import networkx as nx

# plotting stuff
import matplotlib.pyplot as plt
import cartopy

# data exploration
import xarray as xr
import cartopy.crs as ccrs


c:\Users\Carlos\anaconda3\envs\pypsa-earth\lib\site-packages\pypsa\networkclustering.py:16: UserWarning: The namespace `pypsa.networkclustering` is deprecated and will be removed in PyPSA v0.24. Please use `pypsa.clustering.spatial instead`. 
  warnings.warn(


# Load data sets

In [ ]:
# Load profile file from PyPSA-BO (build_renewable_profiles) and shapefile from MSR for costs and capacity caps (3.preescreened) and profiles (2.profile_generator)

pypsa_resources_path = r"C:\Users\Carlos\Desktop\PyPSA-BO\pypsa-earth\resources\renewable_profiles\profile_onwind_old.nc" 
MSR_shapes_path = r"C:\Users\Carlos\Desktop\MSR\Model-Supply-Regions-MSR-Toolset\3. Attributor and ShapeFileCombiner\Results-Prescreen_Bolivia\Wind_prescreen.geojson"
MSR_profiles_path = r"C:\Users\Carlos\Desktop\MSR\Model-Supply-Regions-MSR-Toolset\2. Profile Generator\Output_Bolivia\Results_UTC_Profiles\Bolivia\Bolivia wind 100m BiasCorrected ResourceProfiles.csv"
n_base = pypsa.Network(r"C:\Users\Carlos\Desktop\PyPSA-BO\pypsa-earth\networks\base.nc")
power_curve_path = r"C:\Users\Carlos\Desktop\MSR\Model-Supply-Regions-MSR-Toolset\2. Profile Generator\Wind speed IEC Classes.csv"


resources = xr.open_dataset(pypsa_resources_path)
shapes = gpd.read_file(MSR_shapes_path)
profiles = pd.read_csv(MSR_profiles_path)
df_pc = pd.read_csv(power_curve_path)

c:\Users\Carlos\anaconda3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\Carlos\anaconda3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\Carlos\anaconda3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatibl

# Coupling MSR outputs

In [ ]:
#reorganizing profiles from MSR (script 2)

profile_cols = [c for c in profiles.columns if c.startswith("H")]

df_long = (
    profiles[["MSR_ID"] + profile_cols]
    .melt(id_vars="MSR_ID", var_name="hour", value_name="profile")
)

df_long["time"] = df_long["hour"].str.replace("H", "").astype(int) - 1

#convert to xarray
xr_msr = (
    df_long
    .set_index(["time", "MSR_ID"])
    .to_xarray()
)

In [4]:
#Add parameters from the shapes file into the new xarray 
xr_msr["p_nom_max"] = ("MSR_ID", shapes.set_index("FID")["CapacityMW"])
xr_msr["CAPEX"] = ("MSR_ID", shapes.set_index("FID")["trCAPEX-kW"])
xr_msr["AreakM2"] = ("MSR_ID", shapes.set_index("FID")["AreakM2"])

xr_msr["weight"] = ("MSR_ID", np.ones(len(xr_msr.MSR_ID)))

In [ ]:
#Check coordinated for each MSR
msr_coor = pd.DataFrame()

msr_coor["MSR_ID"] = shapes["FID"]
msr_coor["lon"] = shapes.geometry.centroid.x
msr_coor["lat"] = shapes.geometry.centroid.y

msr_coor = msr_coor.rename({"FID":"MSR_ID"})

#Add coordinates of each MSR
xr_msr = xr_msr.assign_coords(
    lat=("MSR_ID", msr_coor["lat"].values),
    lon=("MSR_ID", msr_coor["lon"].values),)

C:\Users\Carlos\AppData\Local\Temp\ipykernel_18244\1199424797.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  msr_coor["lon"] = shapes.geometry.centroid.x
C:\Users\Carlos\AppData\Local\Temp\ipykernel_18244\1199424797.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  msr_coor["lat"] = shapes.geometry.centroid.y


# Mapping PyPSA buses to MSR map

In [6]:
# Loading bus map from pypsa

buses = n_base.buses.loc[resources.bus.values] #.loc[resources.bus.values] -> this section makes the buses match only the ones available in resources, not all the ones available in all buses

gdf_resources = gpd.GeoDataFrame(
    buses,
    geometry=gpd.points_from_xy(buses.x, buses.y),
    crs="EPSG:4326"
)


In [7]:
# creating MSR map from centroids
gdf_msr_pts = gpd.GeoDataFrame(
    msr_coor,
    geometry=gpd.points_from_xy(msr_coor.lon, msr_coor.lat),
    crs="EPSG:4326"
)

# reprojecting (for distances?)
gdf_msr_pts = gdf_msr_pts.to_crs(3857)
gdf_resources = gdf_resources.to_crs(3857)

#Assign a bus to each MSR
msr_to_bus = gpd.sjoin_nearest(
    gdf_msr_pts,
    gdf_resources[["geometry"]],
    how="left"
)

msr_to_bus = (
    msr_to_bus
    .reset_index(drop=True)
    .rename(columns={"index": "MSR_ID", "index_right": "bus"})
)

#create data set of equivalence MSR-bus
bus_map = (
    msr_to_bus
    .set_index("MSR_ID")
    .loc[xr_msr.MSR_ID.values, "bus"]
    .astype(str)
)

#Add bus as coordinate in the xarray
xr_msr = xr_msr.assign_coords(
    bus=("MSR_ID", bus_map.values)
)

# Aggregate weighted MSR profiles to each (PyPSA) bus 

In [ ]:
#define weight and aggragate based on buses and numerator and denominator for weighted average
weights = xr_msr["AreakM2"]

num = (xr_msr["profile"] * weights).groupby("bus").sum(dim="MSR_ID")

den = weights.groupby("bus").sum(dim="MSR_ID")

xr_bus_profile = num / den
xr_bus_profile

p_nom_bus = xr_msr["p_nom_max"].groupby("bus").sum(dim="MSR_ID")

capex_bus = (
    (xr_msr["CAPEX"] * xr_msr["AreakM2"])
    .groupby("bus")
    .sum(dim="MSR_ID")
    / den
)

<xarray.DataArray (time: 8760, bus: 87)>
array([[ 2.7278204 , 10.482     , 11.678     , ...,  6.88105766,
         3.185     ,  7.42638223],
       [ 7.91130686, 11.175     , 11.355     , ...,  8.67061954,
         7.698     ,  7.67603707],
       [ 5.33867605, 10.685     , 10.87      , ...,  8.6112636 ,
         3.467     ,  6.18548697],
       ...,
       [ 9.5340518 ,  6.114     ,  6.714     , ...,  8.45637938,
         8.207     ,  2.71864432],
       [ 7.9043421 ,  6.778     ,  6.579     , ...,  7.68207403,
         5.574     ,  1.79194861],
       [ 8.46646121,  6.342     ,  7.649     , ...,  7.35276777,
         6.221     ,  2.06707488]])
Coordinates:
  * time     (time) int32 0 1 2 3 4 5 6 7 ... 8753 8754 8755 8756 8757 8758 8759
  * bus      (bus) object '1' '100' '101' '102' '103' ... '94' '95' '96' '97'

# Build equivalent xarray (MSR to PyPSA)

In [10]:
#Attach al previous data sets into a single xarray fule
xr_out = xr.Dataset(
    {
        "profile": xr_bus_profile,
        "p_nom_max": p_nom_bus,
        "CAPEX": capex_bus,
    }
)

xr_out = xr_out.assign_coords(
    x=("bus", n_base.buses.loc[xr_out.bus.values, "x"].values),
    y=("bus", n_base.buses.loc[xr_out.bus.values, "y"].values),
)

#Change time format from integer to date/hour
xr_out = xr_out.assign_coords(
    time=pd.date_range("2013-01-01", periods=8760, freq="H")
)


C:\Users\Carlos\AppData\Local\Temp\ipykernel_18244\2126980947.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time=pd.date_range("2013-01-01", periods=8760, freq="H")


# Create new wind_profile map

In [ ]:
#interpolate windspeeds for the IEC wind classed used
from scipy.interpolate import interp1d

IECclass = "IEC2"
interp_curves = {
    IECclass: interp1d(
        df_pc["Wind speed m/s"],
        df_pc["IEC Class 2"],
        bounds_error=False,
        fill_value=0.0
    ),
}

curve = interp_curves[IECclass]

def apply_power_curve(ws, curve):
    return curve(ws)

xr_new = xr_out.copy()

xr_new["profile_cf"] = xr.apply_ufunc(
    apply_power_curve,
    xr_out["profile"],
    kwargs={"curve": curve},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

xr_out["profile"] = xr_new["profile_cf"]

#ensure datatype consistency between both datasets
resources = resources.assign_coords(
    bus=resources.bus.astype(str)
)

xr_out = xr_out.assign_coords(
    bus=xr_out.bus.astype(str)
)

In [12]:
#overwrite the profiles from atlite with the ones from MSR
resources_new = resources.copy(deep=True)

resources_new["profile"].loc[
    dict(bus=xr_out.bus)
] = xr_out["profile"]

#overwrite the max capacity from atlite with the one from MSR
resources_new["p_nom_max"].loc[
    dict(bus=xr_out.bus)
] = xr_out["p_nom_max"]

#and store capex as a potential future input
if "CAPEX" not in resources_new:
    resources_new["CAPEX"] = xr.zeros_like(resources_new["p_nom_max"])

resources_new["CAPEX"].loc[
    dict(bus=xr_out.bus)
] = xr_out["CAPEX"]

#Set to 0 data for buses not considered in the MSR
non_msr_buses = sorted(
    set(resources_new.bus.values) - set(xr_out.bus.values)
)

resources_new["profile"].loc[
    dict(bus=non_msr_buses)
] = 0.0

resources_new["p_nom_max"].loc[
    dict(bus=non_msr_buses)
] = 0.0

In [13]:
resources_new.to_netcdf("Cost-Supply\profile_onwind_new_test.nc")